# Portfolio Analysis (self-contained)

**Pipeline:** This notebook runs the full factor pipeline (same as `run_all.py`) so you do **not** need to run individual factors or `run_all.py` beforehand. It then loads the combined factor returns, runs scenario analysis, and saves all plots to a single directory.

**Directory structure:**
- Inputs: `src/` (factor modules), `config.py` (weights, params), data files as expected by each factor.
- Outputs: `outputs/factors/` (intermediate factor outputs), `outputs/portfolio_combined/` (factor_returns.xlsx, etc.), `outputs/plots/` (all figures from this notebook).

**Parameters:** Region, default weights, and scenario weights come from `config.py` (e.g. `DEFAULT_WEIGHTS_COMBINED`). You can override scenarios in the CONFIGURATION cell.

## Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)

for _s in ('seaborn-v0_8-darkgrid', 'seaborn-darkgrid', 'ggplot', 'default'):
    try:
        plt.style.use(_s)
        break
    except Exception:
        pass
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

PLOTS_DIR = ROOT / "outputs" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print('✓ Setup complete. Plots will be saved to:', PLOTS_DIR)

## Run full pipeline (no pre-run needed)

In [ ]:
from run_all import run_all_factors, build_factor_matrix

factor_dfs = run_all_factors()
factor_returns = build_factor_matrix(factor_dfs, 'combined')
factor_returns = factor_returns.dropna(how='all')

print(f'✓ Loaded {len(factor_returns)} months. Period: {factor_returns.index[0].strftime("%Y-%m")} to {factor_returns.index[-1].strftime("%Y-%m")}')
print('  Factors:', list(factor_returns.columns))
display(factor_returns.tail())

## Configuration (edit to test scenarios)

In [ ]:
import config

MONTHS_TO_ANALYZE = 6
USE_RECENT = True
START_DATE = "2023-01-01"

# Scenarios: use same factor names as factor_returns (VAL, MOM, LIQ, QLT, LVOL, BETA0, BETA1, BETA2)
default_w = {k: v for k, v in config.DEFAULT_WEIGHTS_COMBINED.items() if k in factor_returns.columns}
equal_w = {c: 1.0 / len(factor_returns.columns) for c in factor_returns.columns}

SCENARIOS = {
    'Default (config)': default_w,
    'Equal Weight': equal_w,
    'Custom': {**default_w, 'VAL': 0.2, 'QLT': 0.2},
}
print('✓ Scenarios:', list(SCENARIOS.keys()))

## Helpers

In [ ]:
def get_period_returns(factor_returns, use_recent=True, start_date=None, months=6):
    if use_recent:
        return factor_returns.iloc[-months:]
    start = pd.to_datetime(start_date)
    end = start + pd.DateOffset(months=months)
    return factor_returns.loc[start:end]

def calculate_portfolio_returns(factor_returns, weights):
    portfolio_ret = pd.Series(0.0, index=factor_returns.index)
    for factor, weight in weights.items():
        if factor in factor_returns.columns and weight != 0:
            portfolio_ret += weight * factor_returns[factor].fillna(0)
    return portfolio_ret

def calculate_metrics(returns):
    total_return = (1 + returns).prod() - 1
    avg_return = returns.mean()
    volatility = returns.std()
    sharpe = (avg_return / volatility * np.sqrt(12)) if volatility > 0 else 0
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = (cum_returns - running_max) / running_max
    max_dd = drawdown.min()
    return {
        'Total Return': total_return * 100,
        'Avg Monthly': avg_return * 100,
        'Volatility': volatility * 100,
        'Sharpe': sharpe,
        'Max DD': max_dd * 100,
    }

## Calculate performance

In [ ]:
period_returns = get_period_returns(factor_returns, USE_RECENT, START_DATE, MONTHS_TO_ANALYZE)
portfolio_returns = {}
for name, weights in SCENARIOS.items():
    portfolio_returns[name] = calculate_portfolio_returns(period_returns, weights)

results = {name: calculate_metrics(ret) for name, ret in portfolio_returns.items()}
results_df = pd.DataFrame(results).T
print('✓ Performance calculated')
display(results_df.round(2))

## Plots (saved to outputs/plots/)

In [ ]:
colors = ['#2E86AB', '#A23B72', '#F18F01']

# 1) Factor returns over time
fig, ax = plt.subplots(figsize=(10, 4))
for col in factor_returns.columns:
    s = factor_returns[col].dropna()
    if len(s) > 0:
        ax.plot(s.index, s, label=col, alpha=0.8)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_ylabel('Monthly return')
ax.set_title('Factor returns over time')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'factor_returns_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

# 2) Cumulative returns (scenarios)
fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, returns) in enumerate(portfolio_returns.items()):
    cum = (1 + returns).cumprod()
    ax.plot(cum.index, cum.values, marker='o', label=name, linewidth=2.5, color=colors[i % len(colors)])
ax.set_title('Cumulative Returns - 6 Month Comparison')
ax.set_xlabel('Month')
ax.set_ylabel('Cumulative Return')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.axhline(1, color='black', linestyle='--', linewidth=1, alpha=0.5)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

# 3) Total return by scenario (bar)
fig, ax = plt.subplots(figsize=(10, 6))
total_returns = {name: calculate_metrics(ret)['Total Return'] for name, ret in portfolio_returns.items()}
scenarios = list(total_returns.keys())
returns_pct = list(total_returns.values())
bars = ax.bar(scenarios, returns_pct, color=colors[:len(scenarios)], alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_title('Total Return by Scenario (%)')
ax.set_ylabel('Return (%)')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linewidth=1)
for bar, val in zip(bars, returns_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.2f}%', ha='center', va='bottom' if val > 0 else 'top', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'total_return_by_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

# 4) Monthly returns by scenario
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(period_returns))
width = 0.25
for i, (name, returns) in enumerate(portfolio_returns.items()):
    offset = (i - len(portfolio_returns)/2 + 0.5) * width
    ax.bar(x + offset, returns.values * 100, width, label=name, alpha=0.8, color=colors[i % len(colors)])
ax.set_title('Monthly Returns by Scenario (%)')
ax.set_xlabel('Month')
ax.set_ylabel('Return (%)')
ax.set_xticks(x)
ax.set_xticklabels([d.strftime('%b %y') for d in period_returns.index], rotation=45, ha='right')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'monthly_returns_by_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

# 5) Factor contributions (first scenario)
first_name = list(SCENARIOS.keys())[0]
first_weights = list(SCENARIOS.values())[0]
contributions = {}
for factor, weight in first_weights.items():
    if factor in period_returns.columns and weight != 0:
        contrib = ((1 + weight * period_returns[factor].fillna(0)).prod() - 1) * 100
        contributions[factor] = contrib
fig, ax = plt.subplots(figsize=(10, 6))
factors = list(contributions.keys())
values = list(contributions.values())
bar_colors = ['#27AE60' if v > 0 else '#E74C3C' for v in values]
ax.barh(factors, values, color=bar_colors, alpha=0.8, edgecolor='black')
ax.set_title(f'Factor Contributions - {first_name}')
ax.set_xlabel('Contribution to Total Return (%)')
ax.axvline(0, color='black', linewidth=1)
ax.grid(True, alpha=0.3, axis='x')
for i, (f, v) in enumerate(zip(factors, values)):
    ax.text(v, i, f' {v:.2f}%', va='center', ha='left' if v > 0 else 'right', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'factor_contributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ All plots saved to', PLOTS_DIR)

In [ ]:
best = max(total_returns.items(), key=lambda x: x[1])
print(f'Best scenario: {best[0]} ({best[1]:.2f}%)')